# # Titanic Dataset - Exploration & Missing Value Analysis

## Objective
Understand dataset structure and handle missing values using multiple strategies.

# Data Loading

In [4]:
import pandas as pd
import numpy as np


df = pd.read_csv('/content/sample_data/train.csv')
df.head()
df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


# Intial Exploration

*   Dataset has 12 columns and 891 rows.
*   Some columns like Age, Cabin and Embarked has missing values.
*   Cabin has high amount of missing values.

# Column Understanding

**PassengerId:**        
Meaning: Every passenger has unique ID.   
Type: Numerical    


**Survived:**        
Meaning: 1 -> Survived, 0 -> Died.      
Type: Numerical

**Pclass:**       
Meaning: Shows Ticket Class -> 1, 2, 3 CLass      
Type: Numerical

**Name:**             
Meaning: Name of every Passenger          
Type: Categorical

**Sex:**          
Meaning: Male or Female           
Type: Categorical

**Age:**     
Meaning: Age of each passenger     
Type: Numerical     
Issue: Has missing values (177).

**SibSp:**     
Meaning: number of siblings/spouses aboard      
Type: Numerical      

**Parch:**    
Meaning: Number of Parents/children aboard     
Type: Numerical

**Ticket:**   
Meaning: Ticket Number      
Type: Numerical

**Fare:**   
Meaning: Ticket Price      
Type: Numerical     

**Cabin:**    
Meaning: Cabin Number        
Type: Categorical     
Issue: Has Missing values (687).

**Embarked:**
Meaning: boarding port       
Type: Categorical      
Issue: Has Missing values (2).



In [5]:
df.head()
df.tail()
df.info()
df.describe()
df.columns
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


(891, 12)

In [6]:
#Cheking for unique values (Useful for spotting typos or unexpected labels.)
df["Sex"].unique()
df["Embarked"].unique()
df["Pclass"].unique()


df["Age"].describe() #Checking for Negative values
df["Fare"].describe()

,Fare
count,891.000000
mean,32.204208
std,49.693429
min,0.000000
25%,7.910400
50%,14.454200
75%,31.000000
max,512.329200


116 passengers have unusually high or low ticket fares         
11 People considered more elderly

In [7]:
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    return df[column][
        (df[column] < Q1 - 1.5 * IQR) |
        (df[column] > Q3 + 1.5 * IQR)
    ]


detect_outliers(df, 'Fare')
#detect_outliers(df, "Age")

,Fare
1,71.2833
27,263.0000
31,146.5208
34,82.1708
52,76.7292
...,...
846,69.5500
849,89.1042
856,164.8667
863,69.5500


# Missing values Analysis and Handling
Cabin has high amount of missing values: 687 -> drop column     
Embarked column has only 2 missing values -> filled with mode      
Age has 177 missing value -> filled with average age     


Handling missing values reduced number of columns by one, since we dropped Cabin column

In [8]:
df.isnull().sum()


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [9]:
df = df.drop("Cabin", axis=1)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Age"] = df["Age"].fillna(df["Age"].mean())

In [12]:
df.shape

(891, 11)

# Survival Analysis
survived people are 342 and there is 549 who died         
According to the mean value survival rate is 38.39%         
First Class has the highest survival rate.         
Kids also has the highest survival rate compared to teens and adults and seniors.       
Females has higher survival rate compared to males on the Titanic.

In [17]:
df['Survived'].value_counts()
print(df['Survived'].mean())

# Number of survivals and moratlies by Sex and Class
df.groupby('Sex')['Survived'].value_counts()
df.groupby('Pclass')['Survived'].value_counts()

#survival rate by sex and class
print(df.groupby('Sex')['Survived'].mean())
print(df.groupby('Pclass')['Survived'].mean())


#Survival rate by age
bin = [0, 12, 18, 35, 60, 80]
label = ["Child", "Teen", "Young Adult", "Adult", "Senior"]

df["AgeGroup"] = pd.cut(df["Age"], bins=bin, labels=label)
df.groupby("AgeGroup")['Survived'].mean()

0.3838383838383838
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64


/tmp/ipykernel_8875/3069044521.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("AgeGroup")['Survived'].mean()


,Survived
AgeGroup,
Child,0.579710
Teen,0.428571
Young Adult,0.353271
Adult,0.400000
Senior,0.227273
